In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import csv
import time
from urllib import response
import requests
import pandas as pd
from typing import List, Dict, Optional

In [ ]:


#Function to slow down requests to avoid rate limit on the API
def fetch_with_retry(url, params, max_retries=5):
    for _ in range(max_retries):
        response = requests.get(url, params=params)
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 429:
            wait_time = int(response.headers.get("Retry-After", 20))
            print(f"Rate limited. Retrying in {wait_time} seconds...")
            time.sleep(wait_time)
        else:
            print(f"Error {response.status_code}: {response.text}")
            break
    return None


def fetch_historical_weather_multiple(
    latitudes: List[float],
    longitudes: List[float],
    start_date: str,
    end_date: str,
    target_variable: str,
    location_names: Optional[List[str]] = None,
) -> Dict[str, pd.DataFrame]:
    """
    Fetch historical weather data for multiple lat/lon pairs, including wind direction.

    Args:
        latitudes: List of latitudes.
        longitudes: List of longitudes.
        start_date: Start date in "YYYY-MM-DD" format.
        end_date: End date in "YYYY-MM-DD" format.
        location_names: Optional names for each location (default: "loc_0", "loc_1", ...).

    Returns:
        Dictionary of DataFrames (key: location name, value: weather data).
    """
    if len(latitudes) != len(longitudes):
        raise ValueError("Latitudes and longitudes must have the same length.")
    
    if location_names is None:
        location_names = [f"loc_{i}" for i in range(len(latitudes))]
    elif len(location_names) != len(latitudes):
        raise ValueError("Location names must match latitudes/longitudes length.")

    csv_path = Path('average_heating_days.csv')
    file_exists = csv_path.exists()

    average_heating_days = {}
    base_url = "https://archive-api.open-meteo.com/v1/archive"
    failed_locations = []
    counter = 1

    for lat, lon, name in zip(latitudes, longitudes, location_names):
        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": start_date,
            "end_date": end_date,
            "daily": target_variable,
        }
        
        response = fetch_with_retry(base_url, params=params,max_retries=5)
        if response is None:         
            print(f"Failed to fetch data for {name} at ({lat}, {lon}). Skipping.")
            failed_locations.append((name, lat, lon))
            if len(failed_locations) >= 4:
                print("Too many failed locations. Stopping further requests.")
                return failed_locations
            continue
        print(f"{counter}. Successfully fetched data for {name} at ({lat}, {lon}).")
        df = pd.DataFrame(response['daily'], columns=['time', target_variable])

        
        heating_days_threshold = 18.0
        df['heating_day'] = df[target_variable] < heating_days_threshold
        average_for_location = df['heating_day'].mean() * 365
        average_heating_days[name] = average_for_location

        # Write to CSV immediately after each location
        with open(csv_path, 'a', newline='') as f:
            writer = csv.writer(f)
            if not file_exists:
                writer.writerow(['CFSAUID', 'average_heating_days'])
                file_exists = True
            writer.writerow([name, average_for_location])

        counter += 1
    return failed_locations

In [ ]:
coordinates_df = pd.read_csv('fsa_centroids.csv')
chunk_size = 50
failed_locations_all = []

for i in range(0, len(coordinates_df), chunk_size):
    locations = coordinates_df['CFSAUID'].tolist()[i:i+chunk_size]
    latitude = coordinates_df['latitude'].tolist()[i:i+chunk_size]
    longitude = coordinates_df['longitude'].tolist()[i:i+chunk_size]
    start_date = "2002-01-01"
    end_date = "2011-12-31"
    target_variable = "temperature_2m_max"
    failed_locations = fetch_historical_weather_multiple(latitude, longitude,
                    start_date, end_date,target_variable =target_variable,
                    location_names=locations)
    failed_locations_all.extend(failed_locations)
    print(f"Chunk {i//chunk_size + 1} completed. Failed locations so far: {len(failed_locations_all)}."
          f"Sleeping to avoid rate limits...")
    time.sleep(300) 


# For testing with a my location
# locations = ['N6G']
# latitude = [42.9668]
# longitude = [-81.3049]



Rate limited. Retrying in 20 seconds...
Rate limited. Retrying in 20 seconds...
